In [11]:
import pandas as pd
import csv

In [12]:
import pandas as pd
from pathlib import Path
import re

pasta_principal = Path(r'/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet')

padrao_nome = r'Nome:\s*(.*)' # Seu regex
padrao_codigo = r'Codigo Estacao:\s*(.*)' # Seu regex
padrao_latitude = r'Latitude:\s*(.*)' # Seu regex
padrao_longitude= r'Longitude:\s*(.*)' # Seu regex
padrao_altitude = r'Altitude:\s*(.*)' # Seu regex
padrao_situacao= r'Situacao:\s*(.*)' # Seu regex
colunas = [
    "Data Medicao", "EVAPORACAO DO PICHE, DIARIA(mm)", "INSOLACAO TOTAL, DIARIO(h)",
    "PRECIPITACAO TOTAL, DIARIO(mm)", "TEMPERATURA MAXIMA, DIARIA(°C)", 
    "TEMPERATURA MEDIA COMPENSADA, DIARIA(°C)", "TEMPERATURA MINIMA, DIARIA(°C)", 
    "UMIDADE RELATIVA DO AR, MEDIA DIARIA(%)", "UMIDADE RELATIVA DO AR, MINIMA DIARIA(%)", 
    "VENTO, VELOCIDADE MEDIA DIARIA(m/s)", "nome_estacao", "codigo_estacao", 
    "latitude_estacao", "longitude_estacao", "altitude_estacao", "situacao_estacao"
]
dados_final = pd.DataFrame(columns=colunas)
print(f"Iniciando busca por '*.csv' em: {pasta_principal}\n")

for arquivo in pasta_principal.rglob('*.csv'):
    try:
        print(f"--- Lendo {arquivo.name} ---")
        arquivo_final = Path(rf'/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet/{arquivo.name}_edit.csv')
        conteudo = arquivo.read_text(encoding='utf-8')
        nome_estacao = re.findall(padrao_nome, conteudo, re.DOTALL)
        codigo_estacao = re.findall(padrao_codigo, conteudo, re.DOTALL)
        latitude_estacao = re.findall(padrao_latitude, conteudo, re.DOTALL)
        longitude_estacao = re.findall(padrao_longitude, conteudo, re.DOTALL)
        altitude_estacao = re.findall(padrao_altitude, conteudo, re.DOTALL)
        situacao_estacao = re.findall(padrao_situacao, conteudo, re.DOTALL)


        with open(arquivo, 'r', encoding='utf-8') as f_entrada:
            leitor = csv.reader(f_entrada)
            linhas = list(leitor)

        linhas_filtradas = [linhas[0]] + linhas[11:]

        with open(arquivo_final, 'w', encoding='utf-8', newline='') as f_saida:
            escritor = csv.writer(f_saida)
            escritor.writerows(linhas_filtradas)

        dados = pd.read_csv(arquivo_final, sep=';', decimal='.', encoding='latin1')
        
        dados['nome_estacao'] = nome_estacao
        dados['codigo_estacao'] = codigo_estacao
        dados['latitude_estacao'] = latitude_estacao
        dados['longitude_estacao'] = longitude_estacao
        dados['altitude_estacao'] = altitude_estacao
        dados['situacao_estacao'] = situacao_estacao
            
        dados_final = pd.concat([dados, dados_final], ignore_index=True)
    except Exception as e:
        print(f"    Erro ao ler o arquivo {arquivo.name}: {e}")
dados_final.to_csv('/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet/dados.csv')

Iniciando busca por '*.csv' em: /home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet

--- Lendo dados_83481_D_2009-01-01_2018-01-31.csv ---
    Erro ao ler o arquivo dados_83481_D_2009-01-01_2018-01-31.csv: Length of values (1) does not match length of index (3318)
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv ---
    Erro ao ler o arquivo dados_83437_D_2009-01-01_2026-04-30.csv: Length of values (1) does not match length of index (6329)
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv_edit.csv ---
    Erro ao ler o arquivo dados_83437_D_2009-01-01_2026-04-30.csv_edit.csv: Length of values (1) does not match length of index (6319)
--- Lendo dados_83692_D_2009-01-01_2026-04-30.csv ---
    Erro ao ler o arquivo dados_83692_D_2009-01-01_2026-04-30.csv: Length of values (1) does not match length of index (6329)
--- Lendo dados_83587_D_2009-01-01_2026-04-30.csv ---
    Erro ao ler o arquivo dados_83587_D_2009-01-01_2026-04-30.csv: Length of values (1) does not match length of index (

In [ ]:
import pandas as pd
from pathlib import Path
import re

pasta_principal = Path(r'/home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet')

# Regex ajustados para capturar apenas o texto da linha (sem o \n no final)
padrao_nome = r'Nome:\s*(.*)'
padrao_codigo = r'Codigo Estacao:\s*(.*)'
padrao_latitude = r'Latitude:\s*(.*)'
padrao_longitude = r'Longitude:\s*(.*)'
padrao_altitude = r'Altitude:\s*(.*)'
padrao_situacao = r'Situacao:\s*(.*)'

# Lista para armazenar os DataFrames de cada arquivo antes do concat (mais eficiente)
lista_dataframes = []

print(f"Iniciando busca por '*.csv' em: {pasta_principal}\n")

for arquivo in pasta_principal.rglob('*.csv'):
    # Evita processar o próprio arquivo de saída caso ele seja gerado na mesma pasta
    if arquivo.name == 'dados.csv':
        continue
        
    try:
        print(f"--- Lendo {arquivo.name} ---")
        
        # 1. Lê os metadados do cabeçalho (as primeiras linhas)
        conteudo = arquivo.read_text(encoding='utf-8')
        
        # re.search busca a primeira ocorrência e o .group(1) pega apenas o texto capturado
        def extrair_metadado(padrao, texto):
            busca = re.search(padrao, texto)
            return busca.group(1).strip() if busca else "Não Encontrado"

        # Use re.search ao invés de re.findall, e REMOVA o re.DOTALL
        busca_nome = re.search(padrao_nome, conteudo)
        busca_codigo = re.search(padrao_codigo, conteudo)
        busca_latitude = re.search(padrao_latitude, conteudo)
        busca_longitude = re.search(padrao_longitude, conteudo)
        busca_altitude = re.search(padrao_altitude, conteudo)
        busca_situacao = re.search(padrao_situacao, conteudo)

        # O .group(1).strip() garante que você pegará apenas o texto daquela linha, sem espaços extras ou \n
        nome_estacao = busca_nome.group(1).strip() if busca_nome else "Não Encontrado"
        codigo_estacao = busca_codigo.group(1).strip() if busca_codigo else "Não Encontrado"
        latitude_estacao = busca_latitude.group(1).strip() if busca_latitude else "Não Encontrado"
        longitude_estacao = busca_longitude.group(1).strip() if busca_longitude else "Não Encontrado"
        altitude_estacao = busca_altitude.group(1).strip() if busca_altitude else "Não Encontrado"
        situacao_estacao = busca_situacao.group(1).strip() if busca_situacao else "Não Encontrado"

        # 2. Carrega os dados pulando as 10 primeiras linhas de metadados
        # Obs: Altere o 'sep' para ',' se o seu arquivo usar vírgula em vez de ponto e vírgula
        dados = pd.read_csv(arquivo, sep=';', decimal='.', encoding='utf-8', skiprows=10)
        
        # Se houver erro de colunas vazias ou desalinhadas por conta do encoding do INMET, 
        # você pode testar encoding='latin-1' se o utf-8 falhar nos dados.
        
        # 3. Adiciona as colunas de metadados preenchendo todas as linhas do arquivo
        dados['nome_estacao'] = nome_estacao
        dados['codigo_estacao'] = codigo_estacao
        dados['latitude_estacao'] = latitude_estacao
        dados['longitude_estacao'] = longitude_estacao
        dados['altitude_estacao'] = altitude_estacao
        dados['situacao_estacao'] = situacao_estacao
            
        # Adiciona o dataframe editado à nossa lista
        lista_dataframes.append(dados)
        
    except Exception as e:
        print(f"    Erro ao ler o arquivo {arquivo.name}: {e}")

# 4. Junta todos os arquivos de uma vez só (muito mais rápido do que concatenar um por um no loop)
if lista_dataframes:
    dados_final = pd.concat(lista_dataframes, ignore_index=True)
    
    # Define o caminho final e salva
    caminho_salvamento = pasta_principal / 'dados.csv'
    dados_final.to_csv(caminho_salvamento, index=False, encoding='utf-8-sig')
    print(f"\nProcesso concluído! Arquivo final salvo em: {caminho_salvamento}")
else:
    print("\nNenhum dado foi processado com sucesso.")

Iniciando busca por '*.csv' em: /home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet

--- Lendo dados_83481_D_2009-01-01_2018-01-31.csv ---
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv ---
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv_edit.csv ---
--- Lendo dados_83692_D_2009-01-01_2026-04-30.csv ---
--- Lendo dados_83587_D_2009-01-01_2026-04-30.csv ---
--- Lendo dados_83437_D_2009-01-01_2026-04-30.csv_edit.csv_edit.csv ---

Processo concluído! Arquivo final salvo em: /home/rafael-luiz/Desktop/IC_gnss/database/dados_inmet/dados.csv


In [5]:
dados = pd.read_csv('inmet.CSV', sep=';', decimal=',', encoding='latin1')

In [6]:
dados.head()

,Data,Hora UTC,"PRECIPITAï¿½ï¿½O TOTAL, HORï¿½RIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSï¿½O ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSï¿½O ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/mï¿½),"TEMPERATURA DO AR - BULBO SECO, HORARIA (ï¿½C)",TEMPERATURA DO PONTO DE ORVALHO (ï¿½C),TEMPERATURA Mï¿½XIMA NA HORA ANT. (AUT) (ï¿½C),TEMPERATURA Mï¿½NIMA NA HORA ANT. (AUT) (ï¿½C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (ï¿½C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (ï¿½C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREï¿½ï¿½O HORARIA (gr) (ï¿½ (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2026/01/01,0000 UTC,0.0,887.7,887.7,887.2,NaN,19.9,18.5,20.8,19.8,18.6,18.1,92.0,85.0,92.0,9.0,4.5,1.1,NaN
1,2026/01/01,0100 UTC,0.0,888.1,888.2,887.7,NaN,18.5,17.7,19.8,18.5,18.4,17.7,95.0,92.0,95.0,305.0,1.8,0.8,NaN
2,2026/01/01,0200 UTC,0.0,888.0,888.2,888.0,NaN,18.2,17.5,18.5,18.1,17.8,17.4,96.0,95.0,96.0,301.0,1.9,1.0,NaN
3,2026/01/01,0300 UTC,0.0,887.6,888.0,887.6,NaN,18.1,17.4,18.4,17.8,17.7,17.1,96.0,96.0,96.0,269.0,1.7,0.6,NaN
4,2026/01/01,0400 UTC,0.0,887.0,887.6,887.0,NaN,18.1,17.5,18.3,17.9,17.7,17.3,96.0,96.0,96.0,180.0,1.7,0.8,NaN
